# 02. 전처리 (방언 문장 표 · 평가셋 · 감정 데이터 분할)

> ⚠️ **CPU 런타임으로 실행하세요.** 압축 해제·json 파싱·오디오 자르기만 합니다.

1. 원본 zip을 Colab 로컬에 압축 해제 (00과 같은 셀, 이미 풀려 있으면 건너뜀)
2. 방언 json 전체 → 문장 단위 표 → `processed/dialect_all.csv`
3. STT 품질 확인 (STT vs 방언 전사 CER)
4. 평가셋: 무관 태그 문장, 경상·충청 연령 분포 맞춤 샘플 → `processed/eval_set.csv` + 16kHz 문장 wav
5. 감정 데이터: 다수결 label·agreement, 층화 80/10/10 분할 → `processed/emotion_meta.csv`
6. 요약 그래프 → `reports/figures/`

처음 실행 시 **2번의 `emotion_tag` 고유값 출력을 보고 설정 셀의 `NEUTRAL_TAG`를 고친 뒤** 평가셋 셀부터 다시 실행하세요.

In [ ]:
# [설정] 경로·샘플 수·태그 — 여기만 수정하면 됩니다
REPO_DIR = "/content/drive/MyDrive/mmdl/multimodal-dialect-emotion"
OUT_DIR = "/content/drive/MyDrive/mmdl/processed"   # 결과 csv 저장 위치 (Drive)
FIG_DIR = f"{REPO_DIR}/reports/figures"
RAW_DIR = "/content/drive/MyDrive/mmdl/data/raw"  # Drive에 올린 원본 zip (데이터셋별 하위 폴더)
LOCAL_DIR = "/content/data"                        # Colab 로컬 압축 해제 위치 (세션이 끝나면 지워짐)

# 원본 zip → 압축 해제 폴더
UNZIP = {
    f"{RAW_DIR}/dialect_gs/gs_wav.zip":   f"{LOCAL_DIR}/gs/wav",
    f"{RAW_DIR}/dialect_gs/gs_label.zip": f"{LOCAL_DIR}/gs/label",
    f"{RAW_DIR}/dialect_cc/cc_wav.zip":   f"{LOCAL_DIR}/cc/wav",
    f"{RAW_DIR}/dialect_cc/cc_label.zip": f"{LOCAL_DIR}/cc/label",
    f"{RAW_DIR}/emotion/emotion.zip":     f"{LOCAL_DIR}/emotion/wav",
}
EMO_CSV_SRC = f"{RAW_DIR}/emotion/emotion.csv"   # csv는 zip이 아니라 복사만
EMO_CSV = f"{LOCAL_DIR}/emotion/emotion.csv"
DIALECT_LABEL_GLOBS = {"gs": f"{LOCAL_DIR}/gs/label/**/*.json", "cc": f"{LOCAL_DIR}/cc/label/**/*.json"}
DIALECT_WAV_DIRS = [f"{LOCAL_DIR}/gs/wav", f"{LOCAL_DIR}/cc/wav"]
EMO_WAV_DIR = f"{LOCAL_DIR}/emotion/wav"
SEG_DIR = f"{OUT_DIR}/segments"    # 평가셋 문장 wav (Drive, 이미 있으면 건너뜀)

NEUTRAL_TAG = "neutral"       # '무관'에 해당하는 emotion_tag 값 ← 고유값 출력 보고 수정
EVAL_REGIONS = ["gs", "cc"]   # 비교할 region(residenceProvince) 값: [경상, 충청] 순서
N_PER_REGION = 1000           # 평가셋 지역별 최대 문장 수
DUR_RANGE = (1.0, 10.0)       # 평가셋 문장 길이(초)
MIN_F0_COUNT = 20             # 평가셋 문장의 최소 유효 F0 개수
H_SAMPLE = 500                # H 태그 매핑 규칙을 고를 때 쓸 json 수
H_MIN_RATE = 0.9              # 매핑 성공률이 이보다 낮으면 h_count = NaN
CER_SAMPLE = 5000             # CER 계산할 문장 수 (순수 파이썬이라 샘플링)
SEED = 42

In [ ]:
# Drive 마운트, 저장소 경로 등록(src 임포트용), 결과 폴더 생성
from google.colab import drive
drive.mount("/content/drive")

import os, sys
sys.path.insert(0, REPO_DIR)
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)

In [ ]:
# 원본 zip을 Colab 로컬에 압축 해제 (폴더 먼저 생성, 이미 풀린 폴더는 건너뜀)
import os, shutil, subprocess
from pathlib import Path
for zp, dest in UNZIP.items():
    done = Path(dest) / ".unzipped"  # 끝까지 풀렸을 때만 남기는 표시 (중간에 끊기면 다시 풂)
    if done.exists():
        print("[skip]", dest)
        continue
    os.makedirs(dest, exist_ok=True)
    print(f"{zp} → {dest}")
    r = subprocess.run(["unzip", "-q", "-o", "-O", "cp949", zp, "-d", dest])
    assert r.returncode <= 1, f"unzip 실패 (exit {r.returncode}): {zp}"  # 1 = 경고만 있음
    done.touch()
os.makedirs(os.path.dirname(EMO_CSV), exist_ok=True)
if not os.path.exists(EMO_CSV):
    shutil.copy(EMO_CSV_SRC, EMO_CSV)
print("압축 해제 완료:", LOCAL_DIR)

In [ ]:
# 한글 폰트 설치·matplotlib 설정, 그림 저장 함수, 라이브러리 임포트
!apt-get -qq install -y fonts-nanum > /dev/null
import glob, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from tqdm.auto import tqdm
from src.data import load_json, dialect_sentences, choose_h_rule, load_emotion, cer

fm.fontManager.addfont("/usr/share/fonts/truetype/nanum/NanumGothic.ttf")
plt.rcParams.update({"font.family": "NanumGothic", "axes.unicode_minus": False,
                     "axes.spines.top": False, "axes.spines.right": False})
REGION_COLORS = dict(zip(EVAL_REGIONS, ["#257de0", "#f47230"]))  # 경상=파랑, 충청=주황 (고정)

def save(fig, name):
    fig.savefig(f"{FIG_DIR}/{name}.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("saved:", f"{FIG_DIR}/{name}.png")

## 1. 방언 문장 표

In [ ]:
# 라벨 json 목록, 일부 샘플로 H 태그 → 문장 매핑 규칙 선택 (성공률 출력)
json_files = {r: sorted(glob.glob(g, recursive=True)) for r, g in DIALECT_LABEL_GLOBS.items()}
print({r: len(fs) for r, fs in json_files.items()})
all_files = [f for fs in json_files.values() for f in fs]
h_docs = [load_json(f) for f in random.Random(SEED).sample(all_files, min(H_SAMPLE, len(all_files)))]
H_RULE = choose_h_rule(h_docs, H_MIN_RATE)
del h_docs

In [ ]:
# 전체 json → 문장 단위 표 (file은 json 파일명 기준 = wav 파일명) → dialect_all.csv
rows = []
for f in tqdm(all_files):
    rows += [dict(r, file=Path(f).stem) for r in dialect_sentences(load_json(f), H_RULE)]
dial = pd.DataFrame(rows)
dial.to_csv(f"{OUT_DIR}/dialect_all.csv", index=False, encoding="utf-8-sig")
print(dial.shape, "→", f"{OUT_DIR}/dialect_all.csv")
print(dial["region"].value_counts(dropna=False))
dial.describe()

In [ ]:
# emotion_tag 고유값과 개수 → '무관'에 해당하는 값을 설정 셀의 NEUTRAL_TAG로 지정
print(dial["emotion_tag"].value_counts(dropna=False))
print("현재 NEUTRAL_TAG =", repr(NEUTRAL_TAG), "| 값에 있음:", NEUTRAL_TAG in set(dial["emotion_tag"]))

## 2. STT 품질 확인

In [ ]:
# STT vs 방언 전사(dialect) CER 평균·분포 (공백 제외 글자 기준). 너무 낮으면 STT가 전사를 베낀 것
has_stt = dial["stt"].fillna("").str.len() > 0
print(f"STT 있는 문장: {has_stt.mean():.1%}")
s = dial[has_stt & dial["dialect"].notna()]
s = s.sample(min(CER_SAMPLE, len(s)), random_state=SEED)
cers = pd.Series([cer(d, t) for d, t in zip(s["dialect"], s["stt"])], index=s.index)
print(cers.describe(percentiles=[.1, .25, .5, .75, .9]).round(3))
print("CER == 0 비율:", round((cers == 0).mean(), 3))
print(cers.groupby(s["region"]).mean().round(3))
if cers.mean() < 0.05:
    print(f"⚠️ 평균 CER {cers.mean():.1%} < 5%: STT가 전사와 거의 같음 → Whisper로 별도 STT 필요")

## 3. 평가셋 (무관 문장, 경상·충청 연령 분포 맞춤)

In [ ]:
# 무관·길이·F0 조건 → 5세 연령대별로 두 지역 중 적은 쪽 수에 맞춰 같은 개수 샘플 (지역별 최대 N_PER_REGION)
tags = set(dial["emotion_tag"].dropna())
assert NEUTRAL_TAG in tags, f"NEUTRAL_TAG={NEUTRAL_TAG!r}가 emotion_tag 값에 없음: {sorted(tags)}"
cand = dial[(dial["emotion_tag"] == NEUTRAL_TAG)
            & dial["duration"].between(*DUR_RANGE)
            & (dial["f0_count"] >= MIN_F0_COUNT)
            & dial["region"].isin(EVAL_REGIONS)
            & dial["age"].notna()].copy()
cand["age_bin"] = (cand["age"] // 5 * 5).astype(int)
counts = cand.groupby(["age_bin", "region"]).size().unstack(fill_value=0).reindex(columns=EVAL_REGIONS, fill_value=0)
common = counts.min(axis=1)                                        # 연령대별로 두 지역 모두 뽑을 수 있는 수
quota = (common * min(1.0, N_PER_REGION / max(common.sum(), 1))).astype(int)  # 내림 → 합계 ≤ N_PER_REGION
eval_set = pd.concat([g.sample(quota[b], random_state=SEED)
                      for (b, r), g in cand.groupby(["age_bin", "region"]) if quota.get(b, 0) > 0],
                     ignore_index=True)
display(counts.assign(quota=quota))
print(eval_set["region"].value_counts())

In [ ]:
# 평가셋 문장 오디오를 잘라 16kHz mono wav로 저장 → seg_path 컬럼 추가 → eval_set.csv
import librosa, soundfile as sf
wav_index = {p.stem: p for d in DIALECT_WAV_DIRS for p in Path(d).rglob("*") if p.suffix.lower() == ".wav"}
missing = set(eval_set["file"]) - wav_index.keys()
assert not missing, f"json에 대응하는 wav 없음 {len(missing)}개: {sorted(missing)[:5]}"
paths = []
for r in tqdm(eval_set.itertuples(), total=len(eval_set)):
    out = Path(SEG_DIR) / r.region / f"{r.file}_{r.sentence_id:g}.wav"
    if not out.exists():
        y, _ = librosa.load(wav_index[r.file], sr=16000, mono=True, offset=r.start, duration=r.duration)
        out.parent.mkdir(parents=True, exist_ok=True)
        sf.write(out, y, 16000)
    paths.append(str(out))
eval_set["seg_path"] = paths
eval_set.to_csv(f"{OUT_DIR}/eval_set.csv", index=False, encoding="utf-8-sig")
print(len(eval_set), "→", f"{OUT_DIR}/eval_set.csv")

## 4. 감정 데이터 (다수결 label, 층화 분할)

In [ ]:
# 감정 csv → 다수결 label·agreement, label 층화 train/val/test 80/10/10 → emotion_meta.csv
from sklearn.model_selection import train_test_split
emo = load_emotion(EMO_CSV, EMO_WAV_DIR)
print(emo.shape, "| label 없음:", emo["label"].isna().sum(), "| wav 없음:", (~emo["wav_path"].map(os.path.exists)).sum())
emo = emo.dropna(subset=["label"]).reset_index(drop=True)
print(emo["agreement"].value_counts().sort_index())
train_idx, rest = train_test_split(emo.index, test_size=0.2, stratify=emo["label"], random_state=SEED)
val_idx, test_idx = train_test_split(rest, test_size=0.5, stratify=emo.loc[rest, "label"], random_state=SEED)
emo["split"] = "train"
emo.loc[val_idx, "split"] = "val"
emo.loc[test_idx, "split"] = "test"
display(pd.crosstab(emo["label"], emo["split"], margins=True)[["train", "val", "test", "All"]])
emo.to_csv(f"{OUT_DIR}/emotion_meta.csv", index=False, encoding="utf-8-sig")
print("→", f"{OUT_DIR}/emotion_meta.csv")

## 5. 요약 그래프

In [ ]:
# 지역별 연령 분포: 무관 후보 문장 전체 vs 평가셋 (평가셋은 두 지역이 겹쳐야 정상)
bins = range(int(cand["age"].min()) // 5 * 5, int(cand["age"].max()) + 6, 5)
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5), sharex=True)
for ax, (title, df) in zip(axes, [("무관 후보 문장", cand), ("평가셋", eval_set)]):
    for r in EVAL_REGIONS:
        ax.hist(df.loc[df["region"] == r, "age"], bins=bins, histtype="step", linewidth=2,
                color=REGION_COLORS[r], label=f"{r} (n={(df['region'] == r).sum()})")
    ax.set_title(title)
    ax.set_xlabel("연령 (녹음연도 - 출생연도)")
    ax.legend(frameon=False)
axes[0].set_ylabel("문장 수")
save(fig, "preprocess_age_by_region")

In [ ]:
# 평가셋 지역별 f0_range_st(5~95 백분위 F0 범위, 반음) 분포
print(eval_set.groupby("region")["f0_range_st"].describe().round(2))
fig, ax = plt.subplots(figsize=(7, 3.5))
for r in EVAL_REGIONS:
    ax.hist(eval_set.loc[eval_set["region"] == r, "f0_range_st"].dropna(), bins=40, density=True,
            histtype="step", linewidth=2, color=REGION_COLORS[r], label=r)
ax.set_xlabel("F0 범위 (반음, 5~95 백분위)")
ax.set_ylabel("밀도")
ax.set_title("평가셋 지역별 F0 범위 분포")
ax.legend(frameon=False)
save(fig, "preprocess_f0_range_by_region")

In [ ]:
# 감정 데이터 다수결 label 분포
label_counts = emo["label"].value_counts()
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.bar(label_counts.index, label_counts.values, color="#257de0", width=0.6)
for x, v in zip(label_counts.index, label_counts.values):
    ax.annotate(f"{v:,}", (x, v), ha="center", va="bottom", fontsize=9)
ax.set_ylabel("발화 수")
ax.set_title("감정 데이터 다수결 label 분포")
save(fig, "preprocess_emotion_label_dist")

---
### ⚠️ 커밋 전에 출력 지우기
셀 출력에 AI Hub 전사 원문(방언·표준어·STT·발화문)이 남습니다. 라이선스상 재배포 금지이므로 저장소에 커밋하기 전에 **수정 → 모든 출력 지우기** 후 저장하세요.